#02_silver_sac.sql

Camada Silver - fact_sac

Este notebook transforma registros de atendimento/manifestação (SAC/SRP) em uma tabela Silver validada, tipada e padronizada para análises de experiência do cliente, qualidade de atendimento e churn clínico.



##Objetivos:
- Tipa campos: datas/timestamps, SLA/tempos, CSAT/NPS
- Padroniza categorias/canal/status (UPPER/TRIM)
- Deduplica por protocolo_id (mais recente por ingestion_ts)
- Aplica regras de qualidade: protocolo_id obrigatório, data_abertura parseável, SLA/tempo_resolucao não negativos, csat e nps dentro de faixas aceitáveis (quando numéricos)
- Persiste idempotente via MERGE (Delta) usando row_hash
- Roda incremental via checkpoint por ingestion_ts

##Estratégia:
- SQL-first com TEMP VIEWs (staging)
- TRY_CAST/TRY_TO_TIMESTAMP para conversões resilientes
- row_number() para dedupe determinístico
- rejects em tabela dedicada com reject_reason
- MERGE para upsert idempotente + checkpoint para incremental

##Chave natural:
protocolo_id

##Contexto

In [0]:
%sql
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_rejects;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Pipeline checkpoint

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_ops.pipeline_checkpoint (
  table_name STRING,
  last_ingestion_ts TIMESTAMP,
  last_run_ts TIMESTAMP,
  status STRING
)
USING DELTA;

##Tabela destino (silver + rejects)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_sac (
  protocolo_id STRING,
  beneficiario_id STRING,
  data_abertura TIMESTAMP,
  canal STRING,
  categoria STRING,
  status STRING,
  sla_horas INT,
  tempo_resolucao_horas INT,
  csat INT,
  nps INT,

  -- derivados úteis
  tempo_resolucao_calc_horas INT,
  flag_tempo_resolucao_inconsistente INT,

  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_sac (
  protocolo_id STRING,
  beneficiario_id STRING,
  data_abertura STRING,
  canal STRING,
  categoria STRING,
  status STRING,
  sla_horas STRING,
  tempo_resolucao_horas STRING,
  csat STRING,
  nps STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Leitura incremental do bronze (via checkpoint)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_sac_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_sac'
)
SELECT *
FROM healthcare_dev.bronze.sac_srp_manifestacoes
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem e padronização

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_sac_typed AS
SELECT
  cast(protocolo_id as string) AS protocolo_id,
  cast(beneficiario_id as string) AS beneficiario_id,

  -- data_abertura: parse timestamp se falhar vira NULL
  try_to_timestamp(data_abertura) AS data_abertura,

  upper(trim(cast(canal as string))) AS canal,
  upper(trim(cast(categoria as string))) AS categoria,
  upper(trim(cast(status as string))) AS status,

  try_cast(sla_horas as int) AS sla_horas,
  try_cast(tempo_resolucao_horas as int) AS tempo_resolucao_horas,
  try_cast(csat as int) AS csat,
  try_cast(nps as int) AS nps,

  ingestion_ts,
  load_id,
  source_file
FROM stg_sac_raw;

##Deduplicação determinística (protocolo_id)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_sac_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY protocolo_id
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_sac_typed
) x
WHERE rn = 1;

##Derivações e regras de qualidade (classificação + motivo)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_sac_classified AS
SELECT
  *,
  -- Se tempo_resolucao_horas vier nulo mas houver SLA, calcula ou mantem nulo.
  CASE
    WHEN tempo_resolucao_horas IS NOT NULL THEN tempo_resolucao_horas
    ELSE NULL
  END AS tempo_resolucao_calc_horas,

  -- tempo_resolucao inconsistente com SLA (ex.: resolveu > SLA)
  CASE
    WHEN sla_horas IS NOT NULL AND tempo_resolucao_horas IS NOT NULL AND tempo_resolucao_horas > sla_horas THEN 1
    ELSE 0
  END AS flag_tempo_resolucao_inconsistente,

  CASE
    WHEN protocolo_id IS NULL OR protocolo_id = '' THEN 'MISSING_PROTOCOLO_ID'
    WHEN data_abertura IS NULL THEN 'INVALID_DATA_ABERTURA'
    WHEN sla_horas IS NOT NULL AND sla_horas < 0 THEN 'NEGATIVE_SLA_HORAS'
    WHEN tempo_resolucao_horas IS NOT NULL AND tempo_resolucao_horas < 0 THEN 'NEGATIVE_TEMPO_RESOLUCAO'
    WHEN csat IS NOT NULL AND (csat < 0 OR csat > 10) THEN 'INVALID_CSAT_RANGE'
    WHEN nps IS NOT NULL AND (nps < 0 OR nps > 10) THEN 'INVALID_NPS_RANGE'
    ELSE NULL
  END AS reject_reason
FROM stg_sac_dedup;

##Persiste rejects

In [0]:
%sql
INSERT INTO healthcare_dev.silver_rejects.fact_sac
SELECT
  protocolo_id,
  beneficiario_id,
  cast(data_abertura as string) AS data_abertura,
  canal,
  categoria,
  status,
  cast(sla_horas as string) AS sla_horas,
  cast(tempo_resolucao_horas as string) AS tempo_resolucao_horas,
  cast(csat as string) AS csat,
  cast(nps as string) AS nps,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_sac_classified
WHERE reject_reason IS NOT NULL;

##Valid + row_hash (MERGE idempotente)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_sac_valid AS
SELECT
  protocolo_id,
  beneficiario_id,
  data_abertura,
  canal,
  categoria,
  status,
  sla_horas,
  tempo_resolucao_horas,
  csat,
  nps,
  tempo_resolucao_calc_horas,
  flag_tempo_resolucao_inconsistente,
  ingestion_ts,
  load_id,
  source_file,
  sha2(concat_ws('||',
    protocolo_id,
    coalesce(beneficiario_id,''),
    coalesce(cast(data_abertura as string),''),
    coalesce(canal,''),
    coalesce(categoria,''),
    coalesce(status,''),
    coalesce(cast(sla_horas as string),''),
    coalesce(cast(tempo_resolucao_horas as string),''),
    coalesce(cast(csat as string),''),
    coalesce(cast(nps as string),''),
    coalesce(cast(tempo_resolucao_calc_horas as string),''),
    coalesce(cast(flag_tempo_resolucao_inconsistente as string),'')
  ), 256) AS row_hash
FROM stg_sac_classified
WHERE reject_reason IS NULL;

##MERGE  na silver

In [0]:
%sql
MERGE INTO healthcare_dev.silver.fact_sac AS tgt
USING stg_sac_valid AS src
ON tgt.protocolo_id = src.protocolo_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Atualiza checkpoint

In [0]:
%sql
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_sac' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status
FROM stg_sac_valid;

##Checa sanidade

In [0]:
%sql
SELECT COUNT(*) AS n_silver_sac
FROM healthcare_dev.silver.fact_sac;

In [0]:
%sql
SELECT COUNT(*) AS n_rejects_sac
FROM healthcare_dev.silver_rejects.fact_sac;

In [0]:
%sql
SELECT reject_reason, COUNT(*) AS qtd
FROM healthcare_dev.silver_rejects.fact_sac
GROUP BY reject_reason
ORDER BY qtd DESC;

##Checagem final

Se contadores derem 0: perfeito (dataset clean)

Se contadores derem > 0: a regra não está batendo

In [0]:
%sql
-- Checa valores fora da faixa
SELECT
  SUM(CASE WHEN try_cast(csat as int) IS NOT NULL AND (try_cast(csat as int) < 0 OR try_cast(csat as int) > 10) THEN 1 ELSE 0 END) AS csat_out_range_bronze,
  SUM(CASE WHEN try_cast(nps as int)  IS NOT NULL AND (try_cast(nps as int)  < 0 OR try_cast(nps as int)  > 10) THEN 1 ELSE 0 END) AS nps_out_range_bronze,
  SUM(CASE WHEN try_cast(sla_horas as int) IS NOT NULL AND try_cast(sla_horas as int) < 0 THEN 1 ELSE 0 END) AS sla_negative_bronze
FROM healthcare_dev.bronze.sac_srp_manifestacoes;